In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="POSIST",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="sales",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.bronze.sales_bills_2 limit 1

In [0]:
%sql
-- alter table fq_dev_catalog.silver.void_details add columns (item_void_name STRING);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_catalog.silver.void_details (
    ng_id STRING,
    business_date DATE,
    deployment_name STRING,
    header_void BOOLEAN,
    header_void_comments STRING,
    kots_void BOOLEAN,
    kots_void_comments STRING,
    item_void STRING,
    item_void_name STRING,
    item_void_comments STRING,
    item_void_quantity BIGINT
)
PARTITIONED BY ( -- Low cardinality columns first
    deployment_name
)
TBLPROPERTIES (
    delta.columnMapping.mode = 'name',
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
);

In [0]:
from pyspark.sql.functions import col, explode, size

def merge_void_details(microbatch_df, batch_id):
    # Deduplicate by ng_id, keeping the latest businessDate
    from pyspark.sql import Window
    from pyspark.sql.functions import row_number, col

    window_spec = Window.partitionBy("ng_id").orderBy(col("businessDate").desc())
    deduped_df = (
        microbatch_df.withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )
    deduped_df.createOrReplaceTempView("void_details_microbatch")

    # Run MERGE INTO for upsert
    microbatch_df.sparkSession.sql("""
        MERGE INTO fq_dev_catalog.silver.void_details AS target
        USING void_details_microbatch AS source
        ON target.ng_id = source.ng_id
        WHEN MATCHED THEN UPDATE SET
            business_date = source.businessDate,
            deployment_name = source.deployment_name,
            header_void = source.isVoid,
            header_void_comments = source.voidComment,
            kots_void_comments = source.kots_comment,
            kots_void = source.kots_isVoid,
            item_void_quantity = source.item_void_quantity,
            item_void = source.item_void,
            item_void_comments = source.item_void_comments,
            item_void_name = source.item_void_name
        WHEN NOT MATCHED THEN INSERT (
            ng_id, business_date, deployment_name, header_void, header_void_comments, kots_void_comments, kots_void,
            item_void_quantity, item_void, item_void_comments, item_void_name
        ) VALUES (
            source.ng_id, source.businessDate, source.deployment_name, source.isVoid, source.voidComment, source.kots_comment, source.kots_isVoid,
            source.item_void_quantity, source.item_void, source.item_void_comments, source.item_void_name
        )
    """)

(
    spark.readStream
        .table("fq_dev_catalog.bronze.sales_bills_2")
        .select(
            col("ng_id"),
            col("businessDate"),
            col("deployment_name"),
            col('isVoid'),
            col('voidComment'),
            explode(col("_kots")).alias("kot")
        )
        .withColumn('item_void', explode(col('kot.deleteHistory')))
        .filter((col("isVoid") == True) | (col("kot.isVoid") == True) | (size(col('kot.deleteHistory')) > 0))
        .select(
            "ng_id",
            "businessDate",
            "deployment_name",
            "isVoid",
            "voidComment",
            col("kot.comment").alias("kots_comment"),
            col("kot.isVoid").alias("kots_isVoid"),
            col("item_void.quantity").alias("item_void_quantity"),
            col("item_void.item").alias("item_void"),
            col("item_void.comment").alias("item_void_comments"),
            col('item_void.name').alias("item_void_name")
        )
        .writeStream
        .foreachBatch(merge_void_details)
        .option('mergeSchema', True) # schema-evolution
        .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_void_details')
        .trigger(availableNow=True)
        .start()
).awaitTermination()

In [0]:
%sql
SELECT 
  count(distinct ng_id) as ng_id_cardinality,
  count(distinct deployment_name) as deployment_name_cardinality,
  count(distinct business_date) as business_date_cardinality,
  count(distinct header_void_comments) as header_void_comments_cardinality,
  count(distinct kots_void_comments) as kots_void_comments_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.void_details

In [0]:
%sql
OPTIMIZE fq_dev_catalog.silver.void_details ZORDER BY (business_date); -- high cardinality non-partitioned column first

In [0]:
%sql
select * from fq_dev_catalog.silver.void_details where item_void is not null limit 1

In [0]:
for query in spark.streams.active:
    query.stop()